<!-- CURRICULUM_HEADER_START -->
<div class="mh"><p class="mh-crumb"><a href="/series/prediction/index.html">Weather, Climate and Prediction</a><span class="sep">·</span>Part 1 · Weather or climate?</p><p class="mh-facts"><span class="lvl lvl-introductory">Introductory</span></p><p class="mh-assumes">Comfort with averages and histograms. Python for the notebooks.</p></div>
<!-- CURRICULUM_HEADER_END -->

::: {.callout-note appearance="simple" icon=false}
## TL;DR

Weather is one sample from a distribution; climate is the distribution itself, conventionally estimated over 30 years. Using 34 years of Seattle daily highs, this module plots one year of weather, the 34-year climatological band, the 2021 heat dome against that band, and a decade-to-decade shift in summer temperatures. Afterwards you will be able to explain why a single cold day cannot refute a warming climate and a single hot day cannot prove it.
:::

"It snowed at the climate conference, so much for global warming!" You have heard some version of this joke, and it is wrong for a precise, checkable reason: **weather** and **climate** are not the same kind of thing measured on the same kind of scale. One is a single roll of the dice. The other is everything you would learn about the dice by rolling them ten thousand times.

Using 34 years of real daily temperature data, we make that distinction concrete rather than definitional.


## The precise definitions

The World Meteorological Organization draws the line like this:

- **Weather** is the state of the atmosphere at a specific time and place: today's high temperature, this hour's rainfall, the wind right now.
- **Climate** is the *statistical distribution* of weather over a long period, conventionally 30 years. Not one number but a whole distribution: an average, a spread and a range of extremes.

A single day's temperature is a **sample** drawn from a distribution. Climate *is* that distribution. Asking "does today's snow disprove climate change?" is like asking "does one coin flip landing heads disprove that the coin is fair?": a question about a single draw, aimed at a claim about the underlying process.


<pre class="mermaid">
flowchart TD
    A["What are you describing?"] --> B{"A specific place and moment in time?"}
    B -->|"Yes -- e.g. today's high, this hour's rain"| C["Weather (one sample)"]
    B -->|"No -- a pattern over about 30 years"| D["Climate (a distribution)"]
    C --> E["Can be extreme, unusual, or perfectly ordinary -- only climate tells you which"]
    D --> F["Average, spread, and extremes -- e.g. the normal range for late June"]
</pre>


In [1]:
import re

import pandas as pd
import requests
import plotly.graph_objects as go
from IPython.display import HTML, display


def show(fig, div_id, height=420):
    """Render a self-contained, interactive Plotly figure (hover, zoom, pan)."""
    fig.update_layout(
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(t=60, b=40, l=50, r=20),
    )
    html = fig.to_html(
        full_html=False,
        include_plotlyjs="cdn",
        default_height=f"{height}px",
        div_id=div_id,
    )
    # Strip the auto-generated SRI hash on the CDN <script> tag -- it's a long
    # base64 string that trips secret scanners as a false positive.
    html = re.sub(r'\s+integrity="[^"]*"\s+crossorigin="anonymous"', "", html)
    display(HTML(html))

## Seattle, 1990–2023

Daily maximum temperature for Seattle, Washington, from January 1990 through December 2023: 34 full years, 12,388 days, from the [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api), which is built on the ECMWF ERA5 reanalysis (a physics-model reconstruction of the historical atmosphere, blended with real observations). We drop leap days purely so every year lines up on the same 365-day calendar axis.


In [2]:
response = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": 47.6062,
        "longitude": -122.3321,
        "start_date": "1990-01-01",
        "end_date": "2023-12-31",
        "daily": "temperature_2m_max",
        "timezone": "America/Los_Angeles",
    },
    timeout=60,
)
daily = response.json()["daily"]
df = pd.DataFrame(
    {"date": pd.to_datetime(daily["time"]), "tmax": daily["temperature_2m_max"]}
)
df = df[~((df["date"].dt.month == 2) & (df["date"].dt.day == 29))].copy()
df["year"] = df["date"].dt.year
df["month_day"] = df["date"].dt.strftime("%m-%d")
df.head()

,date,tmax,year,month_day
0,1990-01-01,6.5,1990,01-01
1,1990-01-02,3.8,1990,01-02
2,1990-01-03,5.1,1990,01-03
3,1990-01-04,8.8,1990,01-04
4,1990-01-05,10.3,1990,01-05


### What weather looks like

One real year, 2023, of daily high temperatures. This is weather: jagged, noisy, no two days alike. With only this picture, predicting next Tuesday's temperature would feel close to hopeless, which matches the chaos in the first module of this series.


In [3]:
one_year = df[df["year"] == 2023]
fig = go.Figure(
    data=go.Scatter(
        x=one_year["date"].dt.strftime("%b %d").tolist(),
        y=one_year["tmax"].tolist(),
        mode="lines",
        line=dict(color="#1b6ca8", width=1.3),
        hovertemplate="%{x}<br>%{y:.1f}°C<extra></extra>",
    )
)
fig.update_layout(
    title="Daily high temperature, Seattle, 2023 -- this is weather",
    xaxis_title="Date",
    yaxis_title="Daily high (°C)",
    xaxis=dict(tickmode="array", tickvals=list(range(0, 365, 30))),
)
show(fig, "weather-2023", height=380)

### What climate looks like

Instead of one year, use all 34. For every calendar day (Jan 1, Jan 2, ... Dec 31) we collect that day's high temperature across every year in the record and compute the average and the 10th–90th percentile range. That average-plus-spread, repeated for every day of the year, *is* Seattle's climate.


In [4]:
climatology = df.groupby("month_day")["tmax"].agg(
    mean="mean",
    p10=lambda x: x.quantile(0.10),
    p90=lambda x: x.quantile(0.90),
)
climatology = climatology.reindex(
    sorted(climatology.index, key=lambda md: (md[:2], md[3:]))
)
day_labels = pd.to_datetime("2001-" + climatology.index).strftime("%b %d")

In [5]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=day_labels.tolist() + day_labels.tolist()[::-1],
        y=climatology["p90"].tolist() + climatology["p10"].tolist()[::-1],
        fill="toself",
        fillcolor="rgba(27,108,168,0.18)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        name="10th-90th percentile range",
    )
)
fig.add_trace(
    go.Scatter(
        x=day_labels.tolist(),
        y=climatology["mean"].tolist(),
        mode="lines",
        line=dict(color="#1b6ca8", width=2.5),
        name="34-year average",
        hovertemplate="%{x}<br>average: %{y:.1f}°C<extra></extra>",
    )
)
fig.update_layout(
    title="34 years of daily highs, collapsed to one calendar year -- this is climate",
    xaxis_title="Calendar day",
    yaxis_title="Daily high (°C)",
    xaxis=dict(tickmode="array", tickvals=list(range(0, 365, 30))),
    legend=dict(orientation="h", y=1.12),
)
show(fig, "climatology-band", height=420)

Notice how *smooth* this is compared with the 2023 weather chart. Any single year is noisy; the average of 34 years is a clean, repeatable seasonal curve. That smoothness is the point: climate is the shape you see once you zoom out far enough to average the noise away.


## Weather inside its climate

Now put them together: the climate band from all 34 years, with one real, dramatic day highlighted. On 28 June 2021, Seattle recorded a daily high of **37.9°C** during the Pacific Northwest heat dome, one of the most extreme, well-documented heat events in the region's recorded history. A rapid-attribution study by World Weather Attribution concluded an event that extreme would have been virtually impossible without human-caused climate change.


In [6]:
heat_dome = df[df["date"] == "2021-06-28"].iloc[0]
normal_for_day = climatology.loc[heat_dome["month_day"]]

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=day_labels.tolist() + day_labels.tolist()[::-1],
        y=climatology["p90"].tolist() + climatology["p10"].tolist()[::-1],
        fill="toself",
        fillcolor="rgba(27,108,168,0.18)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        name="Normal range (10th-90th pct)",
    )
)
fig.add_trace(
    go.Scatter(
        x=day_labels.tolist(),
        y=climatology["mean"].tolist(),
        mode="lines",
        line=dict(color="#1b6ca8", width=2),
        name="34-year average",
        hovertemplate="%{x}<br>average: %{y:.1f}°C<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=[pd.to_datetime("2001-" + heat_dome["month_day"]).strftime("%b %d")],
        y=[heat_dome["tmax"]],
        mode="markers+text",
        marker=dict(size=16, color="crimson", symbol="star"),
        text=["June 28, 2021: 37.9°C"],
        textposition="top center",
        name="2021 heat dome",
        hovertemplate="June 28, 2021<br>%{y:.1f}°C<extra></extra>",
    )
)
fig.update_layout(
    title="One extreme weather day, seen against 34 years of climate",
    xaxis_title="Calendar day",
    yaxis_title="Daily high (°C)",
    xaxis=dict(tickmode="array", tickvals=list(range(0, 365, 30))),
    legend=dict(orientation="h", y=1.15),
)
show(fig, "heat-dome-vs-climatology", height=460)

### Reading this chart

The normal range for late June is roughly 18–29°C, centred around 23°C. The heat dome hit **37.9°C**, far outside even the 90th-percentile edge of what climate says is normal for that date. That is the vocabulary this distinction gives you: the *event* is weather (one extraordinary day); the *fact that it was extraordinary* is a climate statement, because "extraordinary" only means anything relative to a distribution.


### Does climate itself change?

Climate is not frozen either. It is a statistic computed over a rolling multi-decade window, and that statistic can shift over time, which is a completely different question from "what will next Tuesday look like". We compare the climatological summer average from the first 10 years of the record against the most recent 10.


In [7]:
summer = df[df["date"].dt.month.isin([6, 7, 8])].copy()
early = summer[summer["year"].between(1990, 1999)]["tmax"]
recent = summer[summer["year"].between(2014, 2023)]["tmax"]

fig = go.Figure()
fig.add_trace(go.Box(y=early.tolist(), name="1990-1999", marker_color="#1b6ca8"))
fig.add_trace(go.Box(y=recent.tolist(), name="2014-2023", marker_color="#e8702a"))
fig.update_layout(
    title=f"Summer (Jun-Aug) daily highs: {early.mean():.1f}°C avg vs {recent.mean():.1f}°C avg",
    yaxis_title="Daily high (°C)",
)
show(fig, "decade-comparison", height=400)

The distributions overlap heavily: any individual day from the 1990s could easily be warmer than any individual day from the 2010s, because day-to-day *weather* variability is large. But the *climate* (the whole distribution, not any one draw from it) has measurably shifted warmer. Both things are true at once, and mixing them up is the "it snowed, so climate change is fake" error from the opening joke, pointed in the other direction.


### Tying the series together

- **Module 1** showed that a specific day's weather becomes unknowable beyond about two weeks, because the atmosphere is chaotic.
- **Module 2** showed that a simple Markov chain can still forecast the *next* day or two well, by exploiting the short persistence that survives before chaos erases it.
- **This module**: climate is what is left when you stop asking about any specific day and instead describe the full distribution weather is drawn from. It is why a seasonal outlook can say "wetter than normal" with real confidence months out, yet never say what next Tuesday looks like: a seasonal outlook is a claim about the distribution, not a draw from it.


## Key takeaways

- **Weather** is a single sample: the atmosphere's state at one time and place.
- **Climate** is the statistical distribution that sample is drawn from, typically computed over 30 years.
- A single extreme day (like Seattle's 37.9°C heat dome) is a weather event; calling it "extreme" is a climate statement, because it is only meaningful relative to a distribution.
- Climate itself can shift over decades, which is climate change, and that is a separate question from what any single day's weather will be.
- Confusing the two runs in both directions: one cold day does not disprove a warming climate, and one hot day does not prove it. The evidence is always in the distribution, never in a single draw from it.


<!-- CURRICULUM_FOOTER_START -->
<div class="mf"><a class="mf-up" href="/series/prediction/index.html"><span>Series</span>Weather, Climate and Prediction</a><a class="mf-next" href="/blogs/rainfall-predictability/why-cant-we-forecast-rain-six-months-out.html"><span>Next</span>Why can&#x27;t we forecast rain six months out?</a></div>
<!-- CURRICULUM_FOOTER_END -->